# Value at Risk: Parametric, Historical, Monte Carlo

Value at Risk compresses a loss distribution into one number in CHF: the loss that is exceeded
only with a small probability over a fixed horizon. It is a quantile. This notebook computes it
three ways, shows where each breaks, and ends with the property VaR lacks that motivates
expected shortfall (05/02).

## 0. Math link

- [Quantiles](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/12_quantiles.ipynb):
  VaR **is** a quantile of the loss distribution.
- [Normal distribution](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/10_normal_distribution.ipynb)
  and [Cumulative distribution function](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/08_cumulative_distribution_function.ipynb):
  the parametric method inverts the normal CDF.
- [Binomial distribution](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/03_binomial_distribution.ipynb):
  counting VaR exceptions in a backtest.
- Portfolio volatility $\sqrt{w^\top\Sigma w}$ from [04/01 Portfolio variance](../04_portfolio_mathematics/01_portfolio_variance.ipynb).

## 1. Introduction

Volatility answers "how much does this portfolio fluctuate?". Risk managers, regulators and
boards ask a different question: "how much can we lose on a bad day?". Value at Risk answers it
with a statement of the form

> With 99% confidence, the portfolio will not lose more than CHF 1.6 million over the next day.

After this notebook you can:

- define VaR as a quantile, with a confidence level, a horizon and a currency;
- compute it parametrically (normal), from history, and by Monte Carlo;
- explain why the normal method understates tail risk for fat-tailed returns;
- scale VaR across horizons and know when $\sqrt{h}$ fails;
- backtest a VaR model by counting exceptions;
- show that VaR can **penalize** diversification.

## 2. Theory

### Definition

Let $L$ be the loss of a portfolio over a horizon $h$ (in CHF, positive = loss). For a
confidence level $\alpha$ (typically 95%, 99% or 99.5%):

$$
\operatorname{VaR}_\alpha = \inf\{\ell : P(L \le \ell) \ge \alpha\}
$$

the $\alpha$-quantile of the loss distribution. With portfolio value $V$ and return $R$ over the
horizon, $L = -V R$ and

$$
\operatorname{VaR}_\alpha = -V\,q_{1-\alpha}(R)
$$

minus the $(1-\alpha)$-quantile of returns, in CHF. A 99% VaR is the loss exceeded on 1 day in 100.

### Method 1: parametric (variance–covariance)

If $R \sim N(\mu, \sigma^2)$ over the horizon, then $q_{1-\alpha}(R) = \mu - z_\alpha\sigma$ with
$z_\alpha = \Phi^{-1}(\alpha)$, so

$$
\operatorname{VaR}_\alpha = V\,(z_\alpha\,\sigma - \mu) \approx V\,z_\alpha\,\sigma
$$

For daily horizons $\mu$ is tiny compared with $\sigma$ and usually dropped.
$z_{0.95} = 1.645$, $z_{0.99} = 2.326$, $z_{0.995} = 2.576$.

For a portfolio with weights $w$ and covariance $\Sigma$ (module 04): $\sigma_p = \sqrt{w^\top\Sigma w}$.
Because $\sigma_p \le \sum_i |w_i|\sigma_i$, the portfolio VaR is at most the sum of the standalone
VaRs; the gap is the **diversification benefit**.

### Method 2: historical simulation

Apply the last $T$ observed returns (e.g. 250 or 500 days) to **today's** portfolio, and take the
empirical quantile of the resulting losses. No distributional assumption: fat tails and
non-linear dependence present in the window are captured. Weaknesses: only what happened in the
window can happen; with 250 days, the 99% VaR rests on the two or three worst days; the estimate
jumps when an extreme day enters or leaves the window.

### Method 3: Monte Carlo

Specify a model for the risk factors (multivariate normal, Student-$t$, GARCH, ...), simulate many
scenarios, **revalue** the portfolio in each, and take the empirical quantile. Flexible: handles
fat tails, non-linear instruments (options), and long horizons with path dependence. Costs:
model risk and computing time; the result has simulation noise.

### Horizon scaling

If daily returns are independent with mean zero and $\sigma$ scales as $\sqrt{h}$, normal VaR does too:

$$
\operatorname{VaR}_{h\text{-day}} = \sqrt{h}\;\operatorname{VaR}_{1\text{-day}}
$$

(Basel's 10-day VaR was often computed this way.) It fails with autocorrelation, volatility
clustering, or when the position would be cut or hedged within the horizon.

### Fat tails

Equity returns have more probability in the tails than the normal (02/03). The Student-$t$
distribution with $\nu$ degrees of freedom is a common model; scaled to variance $\sigma^2$ its
quantile is $t_\nu^{-1}(\alpha)\,\sigma\sqrt{(\nu-2)/\nu}$. For $\nu = 4$ at 99% this is
$3.75 \cdot 0.707\,\sigma = 2.65\,\sigma$, against $2.33\,\sigma$ for the normal. The further into
the tail, the bigger the gap.

### Backtesting

Count **exceptions**: days on which the realized loss exceeded the VaR. If the model is right,
each day is an exception with probability $1 - \alpha$, independently, so over $n$ days the count
is $\operatorname{Binomial}(n,\ 1 - \alpha)$. For 250 days at 99%, about 2.5 exceptions are
expected. The Basel traffic light: 0–4 exceptions green, 5–9 yellow, 10 or more red.

### What VaR does not tell you

- **How bad** the loss is beyond the quantile: two portfolios with the same VaR can have very
  different tail losses.
- VaR is not **subadditive**: $\operatorname{VaR}(A + B)$ can exceed
  $\operatorname{VaR}(A) + \operatorname{VaR}(B)$ (Example 4). For normal returns it is subadditive;
  for credit-type returns with rare large losses it need not be. Expected shortfall (05/02)
  fixes both issues.

## 3. Worked Examples

### Example 1: a single position

CHF 10,000,000 in Swiss equities, daily volatility 1% (about 16% annual), mean ignored.

| | 95% | 99% |
|---|---|---|
| 1-day VaR | $1.645 \cdot 0.01 \cdot 10{,}000{,}000 =$ CHF 164,500 | $2.326 \cdot 0.01 \cdot 10{,}000{,}000 =$ CHF 232,600 |
| 10-day VaR ($\sqrt{10}$) | CHF 520,100 | CHF 735,700 |

### Example 2: a diversified portfolio

CHF 10,000,000 in the 04/01 three-asset portfolio (40% Swiss equities, 30% global equities, 30%
Swiss bonds), annual volatility 11.1%, so daily $\sigma_p = 11.1\%/\sqrt{252} = 0.70\%$:

$$
\operatorname{VaR}_{99\%} = 2.326 \cdot 0.0070 \cdot 10{,}000{,}000 \approx \text{CHF } 162{,}800
$$

The three standalone 99% VaRs ($2.326 \cdot w_i\sigma_i/\sqrt{252} \cdot V$) add up to about
CHF 190,500. The diversification benefit is about CHF 27,700, or 15%.

### Example 3: historical VaR from a short history

Ten daily returns (%): $+0.4,\ -1.2,\ +0.8,\ -0.3,\ -2.5,\ +1.1,\ -0.6,\ +0.2,\ -1.8,\ +0.5$.
Sorted losses (%): $2.5,\ 1.8,\ 1.2,\ 0.6,\ 0.3, \dots$ The 90% VaR is the loss exceeded on 1 day in
10: between the worst (2.5%) and the second worst (1.8%), depending on the quantile convention.
Small samples make the convention matter; large samples make it irrelevant.

### Example 4: VaR punishes diversification

Two independent zero-coupon bonds, CHF 1,000,000 each, each defaults with probability 4% and then
pays nothing.

- **Each bond alone**: $P(L > 0) = 4\% < 5\%$, so the 95% VaR is **CHF 0**.
- **Both together**: $P(\text{at least one default}) = 1 - 0.96^2 = 7.84\% > 5\%$, so the 95% VaR is
  **CHF 1,000,000**.

$\operatorname{VaR}(A + B) = 1{,}000{,}000 > 0 + 0 = \operatorname{VaR}(A) + \operatorname{VaR}(B)$.
By VaR, the diversified portfolio is riskier than the sum of its parts.

## 4. Finance Application

### Who uses VaR

- **Banks**: trading-book limits and, historically, regulatory capital (Basel 1996: 99%, 10-day
  VaR, multiplied by 3 or more depending on backtesting). Basel's Fundamental Review of the
  Trading Book replaced it with 97.5% expected shortfall for capital, but VaR remains the
  daily limit and backtesting measure.
- **Insurers**: Solvency II uses a 99.5% one-year VaR of own funds; the Swiss Solvency Test
  uses expected shortfall instead.
- **Asset managers and pension funds**: risk reporting, and UCITS funds using the VaR approach
  are limited to a 20% monthly 99% VaR (absolute approach).

### Choosing a method

| | Parametric | Historical | Monte Carlo |
|---|---|---|---|
| Distribution | normal (or other fixed form) | empirical window | any model |
| Fat tails | no (unless $t$ etc.) | yes, if in window | yes, if in model |
| Non-linear positions | poorly (delta only) | yes (full revaluation) | yes (full revaluation) |
| Speed | instant | fast | slow |
| Main weakness | normality | window dependence | model risk, noise |

### The usual misreadings

- "VaR is the maximum loss." It is a loss exceeded **with probability $1 - \alpha$**. On 1% of
  days the loss is larger, and VaR says nothing about how much larger.
- "A 99% VaR is 99% certain." It is the output of a model fitted on limited data; its own
  estimation error can be large (the Python section quantifies it).
- "Lower VaR means safer." Example 4 and option-selling strategies (small losses until a large
  one beyond the quantile) show that a portfolio can lower its VaR by moving risk into the tail.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
trading_days = 252

### Parametric VaR (Examples 1 and 2)

In [ ]:
def parametric_var(value, sigma, alpha, mu=0.0):
    # Normal VaR in currency units, positive = loss
    return value * (stats.norm.ppf(alpha) * sigma - mu)


value = 10_000_000   # CHF
for alpha in [0.95, 0.99]:
    var_1d = parametric_var(value, 0.01, alpha)
    print(f"{alpha:.0%}: 1-day VaR CHF {var_1d:,.0f}, 10-day VaR CHF {var_1d * np.sqrt(10):,.0f}")

In [ ]:
assets = ["Swiss equities", "Global equities", "Swiss bonds"]
vols = np.array([0.16, 0.17, 0.05])                  # annual
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma_annual = np.diag(vols) @ corr @ np.diag(vols)
Sigma_daily = Sigma_annual / trading_days
w = np.array([0.4, 0.3, 0.3])

sigma_p = np.sqrt(w @ Sigma_daily @ w)
var_portfolio = parametric_var(value, sigma_p, 0.99)
var_standalone = parametric_var(value * w, np.sqrt(np.diag(Sigma_daily)), 0.99)

print(f"portfolio daily vol: {sigma_p:.3%}")
print(f"portfolio 99% VaR:   CHF {var_portfolio:,.0f}")
print("standalone 99% VaR: ", {a: f"CHF {v:,.0f}" for a, v in zip(assets, var_standalone)})
print(f"sum of standalone:   CHF {var_standalone.sum():,.0f}")
print(f"diversification:     CHF {var_standalone.sum() - var_portfolio:,.0f} "
      f"({1 - var_portfolio / var_standalone.sum():.0%})")

### A fat-tailed return history

Simulate four years of daily returns for the three assets from a multivariate Student-$t$ with
$\nu = 4$ degrees of freedom, scaled to the same covariance matrix. Same volatilities and
correlations as above, fatter tails.

In [ ]:
def multivariate_t(n, Sigma, df, rng):
    # Normal draws divided by sqrt(chi2/df); rescaled so that Cov = Sigma
    L = np.linalg.cholesky(Sigma * (df - 2) / df)
    z = rng.standard_normal((n, Sigma.shape[0])) @ L.T
    chi2 = rng.chisquare(df, size=(n, 1))
    return z / np.sqrt(chi2 / df)


df_t = 4
n_hist = 4 * trading_days
hist_r = multivariate_t(n_hist, Sigma_daily, df_t, rng)
hist_port_r = hist_r @ w

print(f"sample daily vol (portfolio): {hist_port_r.std(ddof=1):.3%}   (model: {sigma_p:.3%})")
print(f"sample excess kurtosis:       {stats.kurtosis(hist_port_r):.1f}   (normal: 0)")

### Three methods side by side

In [ ]:
def historical_var(value, returns, alpha):
    return -value * np.quantile(returns, 1 - alpha)


def monte_carlo_var(value, w, Sigma, df, alpha, n_sims, rng):
    sims = multivariate_t(n_sims, Sigma, df, rng) @ w
    return -value * np.quantile(sims, 1 - alpha)


rows = []
for alpha in [0.95, 0.99, 0.995]:
    rows.append({
        "confidence": f"{alpha:.1%}",
        "parametric normal": parametric_var(value, hist_port_r.std(ddof=1), alpha),
        "historical (4y)": historical_var(value, hist_port_r, alpha),
        "Monte Carlo t(4)": monte_carlo_var(value, w, Sigma_daily, df_t, alpha, 200_000, rng),
        "exact t(4)": value * stats.t.ppf(alpha, df_t) * sigma_p * np.sqrt((df_t - 2) / df_t),
    })

var_table = pd.DataFrame(rows).set_index("confidence")
var_table.style.format("CHF {:,.0f}")

At 95% the normal VaR is **higher** than the $t$ VaR: a fat-tailed distribution with the same
variance has more mass in the centre and far tails, less in the shoulders. At 99% and 99.5% the
order flips and the normal method understates the loss, by more the further out you go. The historical estimate is noisy: four years
contain only about five days beyond the 99.5% quantile.

### How noisy is historical VaR?

Repeat the four-year history many times and look at the spread of the 99% historical VaR.

In [ ]:
n_repeats = 2_000
hist_vars = np.empty(n_repeats)
for k in range(n_repeats):
    sample = multivariate_t(n_hist, Sigma_daily, df_t, rng) @ w
    hist_vars[k] = historical_var(value, sample, 0.99)

exact_99 = var_table.loc["99.0%", "exact t(4)"]
print(f"true 99% VaR:                 CHF {exact_99:,.0f}")
print(f"historical, mean over trials: CHF {hist_vars.mean():,.0f}")
print(f"5%-95% range:                 CHF {np.quantile(hist_vars, 0.05):,.0f} - CHF {np.quantile(hist_vars, 0.95):,.0f}")

### Does $\sqrt{10}$ scaling work?

Sum ten independent daily $t(4)$ returns and compare the 10-day 99% quantile with $\sqrt{10}$ times
the 1-day quantile.

In [ ]:
n_sims = 200_000
daily_sims = multivariate_t(n_sims * 10, Sigma_daily, df_t, rng) @ w
ten_day = daily_sims.reshape(n_sims, 10).sum(axis=1)

var_1d_mc = -value * np.quantile(daily_sims[:n_sims], 0.01)
var_10d_mc = -value * np.quantile(ten_day, 0.01)
print(f"1-day 99% VaR:              CHF {var_1d_mc:,.0f}")
print(f"sqrt(10) x 1-day:           CHF {var_1d_mc * np.sqrt(10):,.0f}")
print(f"10-day 99% VaR (simulated): CHF {var_10d_mc:,.0f}")
print(f"10-day normal VaR:          CHF {parametric_var(value, sigma_p * np.sqrt(10), 0.99):,.0f}")

Summing independent fat-tailed days pulls the 10-day distribution toward the normal (central
limit theorem), so $\sqrt{10}$ times a fat-tailed 1-day VaR **overstates** the 10-day VaR here.
With volatility clustering (module 09) bad days come in runs and the error goes the other way.

### VaR is not subadditive (Example 4)

In [ ]:
n_sims = 1_000_000
p_default, face = 0.04, 1_000_000
loss_a = face * (rng.random(n_sims) < p_default)
loss_b = face * (rng.random(n_sims) < p_default)

def var_from_losses(losses, alpha):
    return np.quantile(losses, alpha)

def es_from_losses(losses, alpha):
    tail = np.sort(losses)[int(np.floor(alpha * len(losses))):]   # worst (1 - alpha) share
    return tail.mean()

for name, losses in [("bond A", loss_a), ("bond B", loss_b), ("A + B", loss_a + loss_b)]:
    print(f"{name:7s}: P(loss > 0) = {np.mean(losses > 0):.2%}, 95% VaR = CHF {var_from_losses(losses, 0.95):,.0f}, "
          f"95% ES = CHF {es_from_losses(losses, 0.95):,.0f}")

The last column previews 05/02: expected shortfall (the average loss in the worst 5% of cases)
**is** subadditive. It sees the default risk in each bond and rewards combining them.

### Backtesting a historical VaR

A daily return series with a volatility regime change: calm (0.8% daily) for three years, a
one-year crisis (2%), then calm again. Each day, compute the 99% historical VaR from the
previous 252 days (one year) and check whether the next day's loss exceeds it.

In [ ]:
regime_vols = np.concatenate([
    np.full(3 * trading_days, 0.008),
    np.full(trading_days, 0.020),
    np.full(trading_days, 0.008),
])
returns_bt = rng.standard_normal(len(regime_vols)) * regime_vols
window = trading_days

var_bt = np.full(len(returns_bt), np.nan)
for t in range(window, len(returns_bt)):
    var_bt[t] = -np.quantile(returns_bt[t - window:t], 0.01)   # as a return, positive = loss

exceptions = (-returns_bt > var_bt) & ~np.isnan(var_bt)
bt = pd.DataFrame({"return": returns_bt, "VaR": var_bt, "exception": exceptions})
bt["year"] = np.arange(len(bt)) // trading_days + 1

per_year = bt[bt["VaR"].notna()].groupby("year")["exception"].agg(["sum", "count"])
per_year.columns = ["exceptions", "days"]


def traffic_light(k):
    return "green" if k <= 4 else ("yellow" if k <= 9 else "red")


per_year["Basel zone"] = per_year["exceptions"].map(traffic_light)
per_year["P(>= k | model right)"] = [stats.binom.sf(k - 1, n, 0.01) for k, n in zip(per_year["exceptions"], per_year["days"])]
per_year.round(4)

In the crisis year the one-year window still remembers the calm regime, the VaR is too low,
and exceptions pile up. After the crisis the window is full of crisis days, the VaR is too high,
and exceptions vanish. Historical VaR reacts with a lag of up to a full window.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(hist_port_r, bins=80, density=True, alpha=0.6, label="simulated daily returns, t(4)")
x = np.linspace(hist_port_r.min(), hist_port_r.max(), 400)
ax.plot(x, stats.norm.pdf(x, 0, hist_port_r.std(ddof=1)), color="black", label="normal, same σ")

for alpha, style in [(0.99, "--"), (0.995, ":")]:
    ax.axvline(-parametric_var(1, hist_port_r.std(ddof=1), alpha), color="tab:blue", linestyle=style,
               label=f"normal {alpha:.1%} VaR")
    ax.axvline(-historical_var(1, hist_port_r, alpha), color="tab:red", linestyle=style,
               label=f"historical {alpha:.1%} VaR")

ax.set_xlim(-0.045, 0.03)
ax.set_title("Portfolio daily returns: normal vs historical VaR")
ax.set_xlabel("daily return")
ax.set_ylabel("density")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# Left tail on a log scale: how fast does the probability of a loss beyond x fall?
losses = np.sort(-hist_port_r)[::-1]
exceed_prob = np.arange(1, len(losses) + 1) / len(losses)
loss_grid = np.linspace(0.005, 0.045, 200)

plt.figure(figsize=(8, 5))
plt.semilogy(losses, exceed_prob, ".", markersize=4, label="historical (4 years)")
plt.semilogy(loss_grid, stats.norm.sf(loss_grid / sigma_p), label="normal")
t_scale = sigma_p * np.sqrt((df_t - 2) / df_t)
plt.semilogy(loss_grid, stats.t.sf(loss_grid / t_scale, df_t), label="t(4)")
plt.axhline(0.01, color="grey", linestyle=":", label="1% (99% VaR)")
plt.xlim(0, 0.045)
plt.ylim(1e-5, 1)
plt.title("Probability of a daily loss larger than x")
plt.xlabel("loss x (share of portfolio value)")
plt.ylabel("P(loss > x)")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.grid(True, alpha=0.3, which="both")
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
days = np.arange(len(bt))
ax.plot(days, bt["return"], color="grey", linewidth=0.6, label="daily return")
ax.plot(days, -bt["VaR"], color="tab:blue", label="− 99% historical VaR (252-day window)")
ax.scatter(days[exceptions], bt["return"][exceptions], color="tab:red", s=15, zorder=3, label="exception")
ax.axvspan(3 * trading_days, 4 * trading_days, color="tab:red", alpha=0.08, label="crisis regime")
ax.set_title("Backtest: exceptions cluster when volatility rises")
ax.set_xlabel("trading day")
ax.set_ylabel("daily return")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=8)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(hist_vars, bins=60, alpha=0.8)
plt.axvline(exact_99, color="black", linestyle="--", label=f"true 99% VaR CHF {exact_99:,.0f}")
plt.title("99% historical VaR from four years of data, 2,000 repetitions")
plt.xlabel("estimated VaR (CHF)")
plt.ylabel("count")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f}k"))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 7. Exercises

### Basic

1. A CHF 5,000,000 bond portfolio has a daily volatility of 0.3%. Compute its 95% and 99% 1-day
   parametric VaR and its 99% 10-day VaR with $\sqrt{10}$ scaling.
2. Use `historical_var` on the ten returns of Example 3 (as decimals) with a CHF 1,000,000
   portfolio at 90%. Try `np.quantile(..., method="lower")` and `method="higher"` too and explain
   the differences.

### Intermediate

3. For a $t$ distribution with $\nu = 3, 4, 6, 10, 30$ degrees of freedom, scaled to unit
   variance, compute the ratio of its 99% and 99.9% quantiles to the normal ones. What happens as
   $\nu$ grows, and why does it matter more at 99.9%?
4. Interpret: a report says "the fund's 1-day 99% VaR is CHF 2 million". List what this
   statement tells a board member, and three things it does not.

### Advanced

5. Show that parametric normal VaR is subadditive: for any two positions,
   $\operatorname{VaR}(A + B) \le \operatorname{VaR}(A) + \operatorname{VaR}(B)$ (with zero means). Then explain why the
   argument fails for the bonds of Example 4.

## 8. Solutions / Checks

In [ ]:
# 1.
v1, s1 = 5_000_000, 0.003
print(f"1. 95%: CHF {parametric_var(v1, s1, 0.95):,.0f}, 99%: CHF {parametric_var(v1, s1, 0.99):,.0f}, "
      f"99% 10-day: CHF {parametric_var(v1, s1, 0.99) * np.sqrt(10):,.0f}")

# 2.
r10 = np.array([0.4, -1.2, 0.8, -0.3, -2.5, 1.1, -0.6, 0.2, -1.8, 0.5]) / 100
for method in ["linear", "lower", "higher"]:
    q = np.quantile(r10, 0.10, method=method)
    print(f"2. method {method:6s}: 10% quantile {q:+.2%}, 90% VaR CHF {-1_000_000 * q:,.0f}")

# 3.
for nu in [3, 4, 6, 10, 30]:
    scale = np.sqrt((nu - 2) / nu)
    ratios = [stats.t.ppf(a, nu) * scale / stats.norm.ppf(a) for a in (0.99, 0.999)]
    print(f"3. nu = {nu:2d}: t/normal quantile ratio at 99% {ratios[0]:.3f}, at 99.9% {ratios[1]:.3f}")

**2.** NumPy's default (`linear`) interpolates between the two sorted returns around the 10%
point; `lower` takes the observation below and `higher` the one above. With 10 observations the
three answers differ by up to CHF 7,000; with 1,000 they would agree to a few CHF.

**3.** The ratios fall toward 1 as $\nu$ grows (the $t$ converges to the normal). At 99% a $t(4)$
is about 14% above the normal, at 99.9% about 64% above: the further into the tail, the more the
normality assumption understates the loss.

**4.** It tells the board: on a typical day, there is about a 1% chance of losing more than
CHF 2 million, under the model's assumptions. It does **not** tell them (a) how large losses are on
those 1% of days; (b) anything about longer horizons or liquidation time; (c) how reliable the
estimate is (model choice, data window, estimation error); also, it is not a maximum loss.

**5.** With zero means, $\operatorname{VaR} = z_\alpha\sigma$, so subadditivity is $\sigma_{A+B} \le \sigma_A + \sigma_B$.
Since $\sigma_{A+B}^2 = \sigma_A^2 + \sigma_B^2 + 2\rho\sigma_A\sigma_B \le (\sigma_A + \sigma_B)^2$ for $\rho \le 1$, it holds
(the same argument as 03/04). For the bonds, the loss distribution is not normal: it is a point mass
at 0 with a 4% jump. VaR then depends on whether the jump probability is above or below
$1 - \alpha$, not on any standard deviation, and combining two bonds pushes the jump probability over
the 5% threshold.

## 9. Conclusion

- VaR is the $\alpha$-quantile of the loss distribution over a horizon, in currency units.
- Parametric: $V z_\alpha \sigma_p$ with $\sigma_p = \sqrt{w^\top\Sigma w}$; fast, but normal tails are too thin at high confidence.
- Historical: empirical quantile of a window; captures fat tails, but noisy and slow to react.
- Monte Carlo: any model, full revaluation; model risk and simulation noise.
- $\sqrt{h}$ scaling assumes independent, identically distributed days; check it.
- Backtest by counting exceptions against $\operatorname{Binomial}(n, 1-\alpha)$; clusters signal a model that lags.
- VaR says nothing beyond the quantile and is not always subadditive.

Next: [02 Expected shortfall (CVaR)](./02_expected_shortfall.ipynb).